# Hybrid PDF Check

하이브리드 로더 결과를 빠르게 점검하기 위한 노트북입니다.

확인 목표:
- 문서 타입 분포 확인
- `hybrid` 문서가 왜 많은지 샘플 확인
- 특정 PDF에서 페이지별로 `native` / `ocr`가 어떻게 섞였는지 확인

In [3]:
from pathlib import Path
import sys
from collections import Counter
import re

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chroma.data_loader import load_pdfs_as_documents, build_hybrid_document

RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'
RAW_DATA_PATH

PosixPath('/home/sms/openclaw_file/project_debugging/PickCardU/data/raw')

In [2]:
docs = load_pdfs_as_documents(str(RAW_DATA_PATH))
len(docs)

[LOAD] BC_Baro_Clear_Plus.pdf
[LOAD] BC_Baro_KaPick.pdf
[LOAD] BC_BizCorporate.pdf
[LOAD] BC_Biz_AirMoney.pdf
[LOAD] BC_Business_Sky.pdf
[LOAD] BC_Green_v2.pdf
[LOAD] BC_KBank_SIMPLE.pdf
[LOAD] BC_K_FRIST.pdf
[LOAD] BC_ON&OFF.pdf
[LOAD] BC_Shopping&.pdf
[LOAD] NH_AllWonderful.pdf
[LOAD] NH_Namu_NH.pdf
[LOAD] NH_Olbareun_Earth.pdf
[LOAD] NH_Olbareun_FLEX.pdf
[LOAD] NH_Olbareun_OIL&PASS.pdf
[LOAD] NH_SKYPASS.pdf
[LOAD] NH_SOHO_Dasaroi_OIL.pdf
[LOAD] NH_zgm_Self.pdf
[LOAD] NH_zgm_living.pdf
[LOAD] NH_zgm_shopping.pdf
[LOAD] Hana_Everyones_Shinsegae.pdf
[LOAD] Hana_JadeClassic.pdf
[LOAD] Hana_JadeFirst.pdf
[LOAD] Hana_OneMoreNext_Members.pdf


Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 84 0 (offset 0)
Ignoring wrong pointing object 116 0 (offset 0)
Ignoring wrong pointing object 120 0 (offset 0)


[LOAD] Hana_OneStore_1.pdf
[LOAD] Hana_One_More_SOHO.pdf
[LOAD] Hana_SKYPASS_Amex_Platinum.pdf
[LOAD] Hana_Travellog.pdf
[LOAD] Hana_Travellog_SKYPASS.pdf
[LOAD] Hana_WonderCard2.0.pdf
[LOAD] Hyundai_BlueMembers_F_M2_20260323.pdf
[LOAD] Hyundai_D_250827.pdf
[LOAD] Hyundai_H_250827.pdf
[LOAD] Hyundai_M.pdf
[LOAD] Hyundai_O_250827.pdf
[LOAD] Hyundai_OliveYoung_20260323.pdf
[LOAD] Hyundai_S_250827.pdf
[LOAD] Hyundai_T_20260319.pdf
[LOAD] Hyundai_The_Orange_20260330.pdf
[LOAD] Hyundai_The_Red_20260330.pdf
[LOAD] IBK_Bliss5.pdf
[LOAD] IBK_DailyWith.pdf
[LOAD] IBK_Everyday_Joy_Credit.pdf


PdfStreamError('Stream has ended unexpectedly')


[LOAD] IBK_I-Anywhere_Green.pdf


...

PdfStreamError('Stream has ended unexpectedly')


[LOAD] IBK_Point3.8(Credit).pdf


PdfStreamError('Stream has ended unexpectedly')


[LOAD] IBK_i-Mileage.pdf
[LOAD] IBK_iAll.pdf
[LOAD] Kookmin_AlphaOne_20210923.pdf
[LOAD] Kookmin_Coupang_Wow_20250702.pdf
[LOAD] Kookmin_Friend_20210917.pdf
[LOAD] Kookmin_Gaon_Nuri_20240105.pdf
[LOAD] Kookmin_GoodDay_20251127.pdf
[LOAD] Kookmin_K-Pass_20240424.pdf
[LOAD] Kookmin_My_WE_SH_20250102.pdf
[LOAD] Kookmin_Star_B_20210923.pdf
[LOAD] Kookmin_YouthRoad_FunAlpha_20210923.pdf
[LOAD] Kookmin_YouthRoad_SweetSleepAlpha_20210923.pdf
[LOAD] Lotte_DIGILOCA_SKYPASS.pdf
[LOAD] Lotte_Digiloca_Edu.pdf
[LOAD] Lotte_Digiloca_LAS_VEGAS.pdf
[LOAD] Lotte_Digiloca_LONDON.pdf
[LOAD] Lotte_Digiloca_MONACO.pdf
[LOAD] Lotte_Digiloca_PARIS.pdf
[LOAD] Lotte_Digiloca_Pet.pdf
[LOAD] Lotte_Hilton_Honors_Amex_Premium.pdf
[LOAD] Lotte_LOCA_LIKIT_Eat.pdf
[LOAD] Lotte_LOCA_LIKIT_Play.pdf
[LOAD] Lotte_LOCA_LIKIT_Shop.pdf
[LOAD] Lotte_LOKA_LIKIT_1.2.pdf
[LOAD] Lotte_LotteMart&MAXX.pdf
[LOAD] Lotte_LotteMembers_Premium.pdf
[LOAD] Samsung_&_MILEAGE_PLATINUM_(SKYPASS).pdf
[LOAD] Samsung_5_V4.pdf
[LOAD] Samsung_To

106

In [3]:
type_counts = Counter(doc.metadata.get('type', 'unknown') for doc in docs)
type_counts

Counter({'hybrid': 60, 'clean': 46})

In [4]:
summary_rows = []
for doc in docs:
    meta = doc.metadata
    summary_rows.append({
        'card_name': meta.get('card_name'),
        'type': meta.get('type'),
        'quality_score': meta.get('quality_score'),
        'native_quality_score': meta.get('native_quality_score'),
        'ocr_quality_score': meta.get('ocr_quality_score'),
        'ocr_pages': meta.get('ocr_pages'),
        'low_quality_pages': meta.get('low_quality_pages'),
        'quality_status': meta.get('quality_status'),
        'source': meta.get('source'),
    })

summary_rows[:5]

[{'card_name': 'BC_Baro_Clear_Plus',
  'type': 'hybrid',
  'quality_score': 70.0,
  'native_quality_score': 22.24,
  'ocr_quality_score': 85.0,
  'ocr_pages': 2,
  'low_quality_pages': 2,
  'quality_status': 'review_needed',
  'source': 'BC_Baro_Clear_Plus.pdf'},
 {'card_name': 'BC_Baro_KaPick',
  'type': 'clean',
  'quality_score': 100.0,
  'native_quality_score': 100.0,
  'ocr_quality_score': 0.0,
  'ocr_pages': 0,
  'low_quality_pages': 0,
  'quality_status': 'ready',
  'source': 'BC_Baro_KaPick.pdf'},
 {'card_name': 'BC_BizCorporate',
  'type': 'hybrid',
  'quality_score': 83.0,
  'native_quality_score': 28.83,
  'ocr_quality_score': 88.5,
  'ocr_pages': 2,
  'low_quality_pages': 2,
  'quality_status': 'review_needed',
  'source': 'BC_BizCorporate.pdf'},
 {'card_name': 'BC_Biz_AirMoney',
  'type': 'clean',
  'quality_score': 97.24,
  'native_quality_score': 89.72,
  'ocr_quality_score': 0.0,
  'ocr_pages': 0,
  'low_quality_pages': 0,
  'quality_status': 'ready',
  'source': 'BC_Bi

In [5]:
try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df.sort_values(['type', 'ocr_pages', 'quality_score'], ascending=[True, False, True]).head(20))
except ImportError:
    for row in summary_rows[:20]:
        print(row)

{'card_name': 'BC_Baro_Clear_Plus', 'type': 'hybrid', 'quality_score': 70.0, 'native_quality_score': 22.24, 'ocr_quality_score': 85.0, 'ocr_pages': 2, 'low_quality_pages': 2, 'quality_status': 'review_needed', 'source': 'BC_Baro_Clear_Plus.pdf'}
{'card_name': 'BC_Baro_KaPick', 'type': 'clean', 'quality_score': 100.0, 'native_quality_score': 100.0, 'ocr_quality_score': 0.0, 'ocr_pages': 0, 'low_quality_pages': 0, 'quality_status': 'ready', 'source': 'BC_Baro_KaPick.pdf'}
{'card_name': 'BC_BizCorporate', 'type': 'hybrid', 'quality_score': 83.0, 'native_quality_score': 28.83, 'ocr_quality_score': 88.5, 'ocr_pages': 2, 'low_quality_pages': 2, 'quality_status': 'review_needed', 'source': 'BC_BizCorporate.pdf'}
{'card_name': 'BC_Biz_AirMoney', 'type': 'clean', 'quality_score': 97.24, 'native_quality_score': 89.72, 'ocr_quality_score': 0.0, 'ocr_pages': 0, 'low_quality_pages': 0, 'quality_status': 'ready', 'source': 'BC_Biz_AirMoney.pdf'}
{'card_name': 'BC_Business_Sky', 'type': 'hybrid', 'qu

In [6]:
hybrid_docs = [doc for doc in docs if doc.metadata.get('type') == 'hybrid']
len(hybrid_docs)

60

In [7]:
def page_method_counts(doc):
    matches = re.findall(r'^\[page\s+\d+\]\[(native|ocr)\]', doc.page_content, re.MULTILINE)
    return Counter(matches)

for doc in hybrid_docs[:5]:
    print('=' * 100)
    print(doc.metadata.get('source'))
    print(doc.metadata)
    print('page methods:', page_method_counts(doc))
    print(doc.page_content[:1200])
    print()

BC_Baro_Clear_Plus.pdf
{'source': 'BC_Baro_Clear_Plus.pdf', 'file_path': '/home/sms/openclaw_file/project_debugging/PickCardU/data/raw/BC/BC_Baro_Clear_Plus.pdf', 'card_name': 'BC_Baro_Clear_Plus', 'total_pages': 2, 'type': 'hybrid', 'card_company': 'BC', 'quality_score': 70.0, 'native_quality_score': 22.24, 'ocr_quality_score': 85.0, 'ocr_pages': 2, 'low_quality_pages': 2, 'quality_status': 'review_needed', 'review_flags': ['amount_risk', 'numeric_risk'], 'review_matches': ['1.1%6', '0.290', '20,.000원'], 'review_count': 3, 'penalty_flags': ['brand_distortion', 'odd_spacing', 'odd_word_form'], 'penalty_matches': ['비로키C', '카드시논부가서비스률변경할 수있습니다', '부가서비스률', '계약올', '반영원', '부과하능'], 'penalty_count': 6, 'penalty_score': 30.0}
page methods: Counter({'ocr': 2})
[page 1][ocr]
비로키C 부가서비스 안내 연회비 안내 카드 이용 시 제공되논 모인트 및 할인혜택 등의 부가서비스는 카드 신규 출시(2022년 4월 1일) 이후 3년 이상 축소 폐지없이 유지I니다. 구분 브랜드 총연회비 기본연회비 제유연회비 상기메도불구하고다음과감은사유가발생한경우카드시논부가서비스률변경할 수있습니다 국내전용 BC 5천원 없음 5천원 카드사의 휴업 파산. 경영상의 위기 등에 따른 불가피한 경우 0의2 제

In [8]:
# 보고 싶은 PDF 파일명을 일부 또는 전체로 넣어서 확인하세요.
target_name = ''

matches = [doc for doc in docs if target_name and target_name in doc.metadata.get('source', '')]
len(matches)

0

In [9]:
if matches:
    doc = matches[0]
    print(doc.metadata)
    print('page methods:', page_method_counts(doc))
    print(doc.page_content[:4000])
else:
    print('target_name 값을 넣고 다시 실행하세요.')

target_name 값을 넣고 다시 실행하세요.


In [10]:
# 특정 raw PDF 하나를 다시 직접 하이브리드 문서로 만들어 확인할 때 사용합니다.
# 예: sample_pdf = RAW_DATA_PATH / 'BC' / 'BC카드_Business_Sky카드.pdf'
sample_pdf = None

if sample_pdf:
    sample_doc = build_hybrid_document(str(sample_pdf))
    print(sample_doc.metadata)
    print('page methods:', page_method_counts(sample_doc))
    print(sample_doc.page_content[:4000])
else:
    print('sample_pdf 경로를 지정하세요.')

sample_pdf 경로를 지정하세요.


## Quality Check

기존 `docs`를 다시 로드하지 않고 품질 점수 세부 항목을 확인합니다.

확인 목표:
- `quality_utils.evaluate_text_quality()` 결과 세부 항목 확인
- 어떤 문서가 왜 높은 점수를 받는지 확인
- OCR 오탈자가 많아도 점수가 높게 나오는 패턴 파악

In [11]:
from chroma.quality_utils import evaluate_text_quality, normalize_extracted_text

In [12]:
def build_quality_row(doc):
    metrics = evaluate_text_quality(doc.page_content)
    row = {
        'source': doc.metadata.get('source'),
        'type': doc.metadata.get('type'),
        'page_methods': dict(page_method_counts(doc)),
        'quality_score': metrics['score'],
        'length': metrics['length'],
        'korean_ratio': metrics['korean_ratio'],
        'alnum_ratio': metrics['alnum_ratio'],
        'broken_ratio': metrics['broken_ratio'],
        'keyword_hits': metrics['keyword_hits'],
        'amount_hits': metrics['amount_hits'],
    }
    return row

In [13]:
quality_rows = [build_quality_row(doc) for doc in docs]
quality_rows[:3]

[{'source': 'BC_Baro_Clear_Plus.pdf',
  'type': 'hybrid',
  'page_methods': {'ocr': 2},
  'quality_score': 70.0,
  'length': 5370.0,
  'korean_ratio': 0.6685,
  'alnum_ratio': 0.7436,
  'broken_ratio': 0.0,
  'keyword_hits': 11.0,
  'amount_hits': 30.0},
 {'source': 'BC_Baro_KaPick.pdf',
  'type': 'clean',
  'page_methods': {'native': 2},
  'quality_score': 100.0,
  'length': 5857.0,
  'korean_ratio': 0.6309,
  'alnum_ratio': 0.6896,
  'broken_ratio': 0.0,
  'keyword_hits': 9.0,
  'amount_hits': 33.0},
 {'source': 'BC_BizCorporate.pdf',
  'type': 'hybrid',
  'page_methods': {'ocr': 2},
  'quality_score': 83.0,
  'length': 4911.0,
  'korean_ratio': 0.6587,
  'alnum_ratio': 0.7361,
  'broken_ratio': 0.0,
  'keyword_hits': 9.0,
  'amount_hits': 19.0}]

In [14]:
try:
    import pandas as pd
    quality_df = pd.DataFrame(quality_rows)
    display(quality_df.sort_values(['quality_score', 'broken_ratio'], ascending=[False, False]).head(20))
except ImportError:
    for row in quality_rows[:20]:
        print(row)

{'source': 'BC_Baro_Clear_Plus.pdf', 'type': 'hybrid', 'page_methods': {'ocr': 2}, 'quality_score': 70.0, 'length': 5370.0, 'korean_ratio': 0.6685, 'alnum_ratio': 0.7436, 'broken_ratio': 0.0, 'keyword_hits': 11.0, 'amount_hits': 30.0}
{'source': 'BC_Baro_KaPick.pdf', 'type': 'clean', 'page_methods': {'native': 2}, 'quality_score': 100.0, 'length': 5857.0, 'korean_ratio': 0.6309, 'alnum_ratio': 0.6896, 'broken_ratio': 0.0, 'keyword_hits': 9.0, 'amount_hits': 33.0}
{'source': 'BC_BizCorporate.pdf', 'type': 'hybrid', 'page_methods': {'ocr': 2}, 'quality_score': 83.0, 'length': 4911.0, 'korean_ratio': 0.6587, 'alnum_ratio': 0.7361, 'broken_ratio': 0.0, 'keyword_hits': 9.0, 'amount_hits': 19.0}
{'source': 'BC_Biz_AirMoney.pdf', 'type': 'clean', 'page_methods': {'native': 2}, 'quality_score': 97.24, 'length': 4986.0, 'korean_ratio': 0.5969, 'alnum_ratio': 0.6978, 'broken_ratio': 0.0022, 'keyword_hits': 9.0, 'amount_hits': 13.0}
{'source': 'BC_Business_Sky.pdf', 'type': 'hybrid', 'page_method

In [15]:
suspicious_docs = [
    doc for doc in docs
    if doc.metadata.get('type') == 'hybrid' and doc.metadata.get('ocr_pages', 0) > 0
]
len(suspicious_docs)

60

In [16]:
for doc in suspicious_docs[:5]:
    metrics = evaluate_text_quality(doc.page_content)
    print('=' * 100)
    print(doc.metadata.get('source'))
    print('metadata:', doc.metadata)
    print('quality metrics:', metrics)
    print('page methods:', page_method_counts(doc))
    print(doc.page_content[:1500])
    print()

BC_Baro_Clear_Plus.pdf
metadata: {'source': 'BC_Baro_Clear_Plus.pdf', 'file_path': '/home/sms/openclaw_file/project_debugging/PickCardU/data/raw/BC/BC_Baro_Clear_Plus.pdf', 'card_name': 'BC_Baro_Clear_Plus', 'total_pages': 2, 'type': 'hybrid', 'card_company': 'BC', 'quality_score': 70.0, 'native_quality_score': 22.24, 'ocr_quality_score': 85.0, 'ocr_pages': 2, 'low_quality_pages': 2, 'quality_status': 'review_needed', 'review_flags': ['amount_risk', 'numeric_risk'], 'review_matches': ['1.1%6', '0.290', '20,.000원'], 'review_count': 3, 'penalty_flags': ['brand_distortion', 'odd_spacing', 'odd_word_form'], 'penalty_matches': ['비로키C', '카드시논부가서비스률변경할 수있습니다', '부가서비스률', '계약올', '반영원', '부과하능'], 'penalty_count': 6, 'penalty_score': 30.0}
quality metrics: {'score': 70.0, 'length': 5370.0, 'korean_ratio': 0.6685, 'alnum_ratio': 0.7436, 'broken_ratio': 0.0, 'keyword_hits': 11.0, 'amount_hits': 30.0, 'penalty_score': 30.0, 'penalty_flags': ['brand_distortion', 'odd_spacing', 'odd_word_form'], 'penal

In [17]:
quality_target_name = ''

quality_matches = [doc for doc in docs if quality_target_name and quality_target_name in doc.metadata.get('source', '')]
len(quality_matches)

0

In [18]:
if quality_matches:
    doc = quality_matches[0]
    print(doc.metadata)
    print('page methods:', page_method_counts(doc))
    print('quality metrics:', evaluate_text_quality(doc.page_content))
    print(doc.page_content[:4000])
else:
    print('quality_target_name 값을 넣고 다시 실행하세요.')

quality_target_name 값을 넣고 다시 실행하세요.


In [19]:
sample_text = ''

if sample_text:
    normalized = normalize_extracted_text(sample_text)
    print(normalized)
    print(evaluate_text_quality(sample_text))
else:
    print('sample_text 값을 넣고 다시 실행하세요.')

sample_text 값을 넣고 다시 실행하세요.


## OCR Error Pattern Classification

기존 `docs`와 `suspicious_docs`를 재사용해서 OCR 오류를 `교정 / 감점 / 검수` 대상으로 분류합니다.

사용 방법:
- `pattern_candidates` 셀에서 샘플 문서를 더 보며 반복 오탈자를 찾습니다.
- `correction_candidates`, `penalty_candidates`, `review_candidates` 리스트를 직접 채워 넣습니다.
- 마지막 요약 셀에서 현재 분류 상태를 한 번에 확인합니다.

In [20]:
pattern_candidates = []

for doc in suspicious_docs[:10]:
    pattern_candidates.append({
        'source': doc.metadata.get('source'),
        'type': doc.metadata.get('type'),
        'page_methods': dict(page_method_counts(doc)),
        'quality_score': doc.metadata.get('quality_score'),
        'preview': doc.page_content[:1200],
    })

pattern_candidates

[{'source': 'BC_Baro_Clear_Plus.pdf',
  'type': 'hybrid',
  'page_methods': {'ocr': 2},
  'quality_score': 70.0,
  'preview': '[page 1][ocr]\n비로키C 부가서비스 안내 연회비 안내 카드 이용 시 제공되논 모인트 및 할인혜택 등의 부가서비스는 카드 신규 출시(2022년 4월 1일) 이후 3년 이상 축소 폐지없이 유지I니다. 구분 브랜드 총연회비 기본연회비 제유연회비 상기메도불구하고다음과감은사유가발생한경우카드시논부가서비스률변경할 수있습니다 국내전용 BC 5천원 없음 5천원 카드사의 휴업 파산. 경영상의 위기 등에 따른 불가피한 경우 0의2 제류업체의휴업 파산 경영상의위기로인해불가피하게부가서비스 축소 변경 하는 경우로서 다른 제휴업체름 통해 동종의 유사한 부가서비스 제공이 불가한 경우 본인 해외경용 MasterCard 5천원 없음 5천원 제휴업체가 카드사의 의사에 반하여 해당 부가서비스 축소하거나 변경 시 당초 부가서비스에 상응하는 다른 부가서비스 제공하는 경우 해외경용 VISA 5천원 없음 5천원 부가서비스틀 3년 이상 제공한 상태에서 해당 부가서비스로 인해 상품의 수익성이 현저히 낮아진 경우 가족카드는 별도의 연회비가 없습니다. 카드사가부가서비스 변경하는 경우 변경사유 변경 내용등올 사유발생 즉시아래고지 VISA 브랜드는 2023년 10월 2일부로 신규발급이 중단되없습니다. 방법 중2가지이상의 방법으로 고지하여 드럽니다 특히부가서비스틀 3년 이상 제공한 상태에서 해당 부가서비스로인해상품의수익성이 현저히낮아저부가서비스 변경하는 고객센터 1899-7771 함페이지 WWW bccardcom 경우에는 6개월 전부터 아래 고지방법 중 2가지이상의방법으로 매월 고지하여드럽니다 [고지방법) 서면교부 우편또는전자우편 전화또는맥스 휴대온메시지또는이에준하는전자적의사표시 해외 이용 안내 연회비 반환 조건 안내 국제브랜드 수수료:%(MasterCard) 1.1%6(VISA)

In [21]:
# 반복 오탈자 중 자동 치환이 가능해 보이는 후보
correction_candidates = [
    {'wrong': '유호기간', 'right': '유효기간', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
    {'wrong': '가맣점', 'right': '가맹점', 'example_source': 'BC_BizCorporate.pdf'},
    {'wrong': '영향울', 'right': '영향을', 'example_source': 'BC_Business_Sky.pdf'},
    {'wrong': '카드틀', 'right': '카드를', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
    {'wrong': '되니다', 'right': '됩니다', 'example_source': 'BC_BizCorporate.pdf'},
    {'wrong': '활수', 'right': '할 수', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
    {'wrong': '금웅', 'right': '금융', 'example_source': 'BC_KBank_SIMPLE.pdf'},
    {'wrong': '실식', 'right': '실적', 'example_source': 'BC_Green_v2.pdf'},
]

correction_candidates

[{'wrong': '유호기간',
  'right': '유효기간',
  'example_source': 'BC_Baro_Clear_Plus.pdf'},
 {'wrong': '가맣점', 'right': '가맹점', 'example_source': 'BC_BizCorporate.pdf'},
 {'wrong': '영향울', 'right': '영향을', 'example_source': 'BC_Business_Sky.pdf'},
 {'wrong': '카드틀', 'right': '카드를', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
 {'wrong': '되니다', 'right': '됩니다', 'example_source': 'BC_BizCorporate.pdf'},
 {'wrong': '활수', 'right': '할 수', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
 {'wrong': '금웅', 'right': '금융', 'example_source': 'BC_KBank_SIMPLE.pdf'},
 {'wrong': '실식', 'right': '실적', 'example_source': 'BC_Green_v2.pdf'}]

In [22]:
# 자동 치환은 위험하지만 품질 점수에서 감점해야 할 패턴 후보
penalty_candidates = [
    {'pattern_type': 'brand_distortion', 'example': '비로키C', 'reason': '브랜드명 심한 왜곡'},
    {'pattern_type': 'brand_distortion', 'example': 'Kbonk', 'reason': '브랜드명 심한 왜곡'},
    {'pattern_type': 'brand_distortion', 'example': 'Srccrfcard', 'reason': '브랜드/상품명 심한 왜곡'},
    {'pattern_type': 'odd_spacing', 'example': '카드시논부가서비스률변경활수있습니다', 'reason': '과도한 붙여쓰기와 오탈자 결합'},
    {'pattern_type': 'odd_word_form', 'example': '가계지금대출금라', 'reason': '비정상 단어 조합'},
    {'pattern_type': 'odd_word_form', 'example': '부가서비스률', 'reason': '비정상 조사/어미 또는 명사 결합'},
]

penalty_candidates

[{'pattern_type': 'brand_distortion',
  'example': '비로키C',
  'reason': '브랜드명 심한 왜곡'},
 {'pattern_type': 'brand_distortion',
  'example': 'Kbonk',
  'reason': '브랜드명 심한 왜곡'},
 {'pattern_type': 'brand_distortion',
  'example': 'Srccrfcard',
  'reason': '브랜드/상품명 심한 왜곡'},
 {'pattern_type': 'odd_spacing',
  'example': '카드시논부가서비스률변경활수있습니다',
  'reason': '과도한 붙여쓰기와 오탈자 결합'},
 {'pattern_type': 'odd_word_form',
  'example': '가계지금대출금라',
  'reason': '비정상 단어 조합'},
 {'pattern_type': 'odd_word_form',
  'example': '부가서비스률',
  'reason': '비정상 조사/어미 또는 명사 결합'}]

In [23]:
# 숫자/금액/퍼센트처럼 자동 수정이 위험해서 사람이 검수해야 하는 후보
review_candidates = [
    {'pattern_type': 'numeric_risk', 'example': '1.1%6', 'reason': '퍼센트 표기 이상'},
    {'pattern_type': 'numeric_risk', 'example': '20P6', 'reason': '숫자/기호 오인식'},
    {'pattern_type': 'amount_risk', 'example': '20,.000원', 'reason': '금액 표기 이상'},
    {'pattern_type': 'numeric_risk', 'example': '39dp', 'reason': '숫자와 영문 기호 혼합 오인식'},
    {'pattern_type': 'numeric_risk', 'example': '089', 'reason': '백분율 또는 소수 표기 오인식 가능성'},
    {'pattern_type': 'numeric_risk', 'example': '0.290', 'reason': '퍼센트 또는 수수료 표기 오인식 가능성'},
]

review_candidates

[{'pattern_type': 'numeric_risk', 'example': '1.1%6', 'reason': '퍼센트 표기 이상'},
 {'pattern_type': 'numeric_risk', 'example': '20P6', 'reason': '숫자/기호 오인식'},
 {'pattern_type': 'amount_risk', 'example': '20,.000원', 'reason': '금액 표기 이상'},
 {'pattern_type': 'numeric_risk',
  'example': '39dp',
  'reason': '숫자와 영문 기호 혼합 오인식'},
 {'pattern_type': 'numeric_risk',
  'example': '089',
  'reason': '백분율 또는 소수 표기 오인식 가능성'},
 {'pattern_type': 'numeric_risk',
  'example': '0.290',
  'reason': '퍼센트 또는 수수료 표기 오인식 가능성'}]

In [24]:
classification_summary = {
    'correction_count': len(correction_candidates),
    'penalty_count': len(penalty_candidates),
    'review_count': len(review_candidates),
    'correction_candidates': correction_candidates,
    'penalty_candidates': penalty_candidates,
    'review_candidates': review_candidates,
}

classification_summary

{'correction_count': 8,
 'penalty_count': 6,
 'review_count': 6,
 'correction_candidates': [{'wrong': '유호기간',
   'right': '유효기간',
   'example_source': 'BC_Baro_Clear_Plus.pdf'},
  {'wrong': '가맣점', 'right': '가맹점', 'example_source': 'BC_BizCorporate.pdf'},
  {'wrong': '영향울', 'right': '영향을', 'example_source': 'BC_Business_Sky.pdf'},
  {'wrong': '카드틀', 'right': '카드를', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
  {'wrong': '되니다', 'right': '됩니다', 'example_source': 'BC_BizCorporate.pdf'},
  {'wrong': '활수', 'right': '할 수', 'example_source': 'BC_Baro_Clear_Plus.pdf'},
  {'wrong': '금웅', 'right': '금융', 'example_source': 'BC_KBank_SIMPLE.pdf'},
  {'wrong': '실식', 'right': '실적', 'example_source': 'BC_Green_v2.pdf'}],
 'penalty_candidates': [{'pattern_type': 'brand_distortion',
   'example': '비로키C',
   'reason': '브랜드명 심한 왜곡'},
  {'pattern_type': 'brand_distortion',
   'example': 'Kbonk',
   'reason': '브랜드명 심한 왜곡'},
  {'pattern_type': 'brand_distortion',
   'example': 'Srccrfcard',
   'reason': '브랜드

In [25]:
# 특정 PDF를 골라 패턴 분류 작업을 반복할 때 사용
pattern_target_name = 'BC_Baro_Clear_Plus.pdf'

pattern_matches = [doc for doc in suspicious_docs if pattern_target_name and pattern_target_name in doc.metadata.get('source', '')]
len(pattern_matches)

1

In [26]:
if pattern_matches:
    doc = pattern_matches[0]
    print(doc.metadata)
    print('page methods:', page_method_counts(doc))
    print(doc.page_content[:5000])
else:
    print('pattern_target_name 값을 넣고 다시 실행하세요.')

{'source': 'BC_Baro_Clear_Plus.pdf', 'file_path': '/home/sms/openclaw_file/project_debugging/PickCardU/data/raw/BC/BC_Baro_Clear_Plus.pdf', 'card_name': 'BC_Baro_Clear_Plus', 'total_pages': 2, 'type': 'hybrid', 'card_company': 'BC', 'quality_score': 70.0, 'native_quality_score': 22.24, 'ocr_quality_score': 85.0, 'ocr_pages': 2, 'low_quality_pages': 2, 'quality_status': 'review_needed', 'review_flags': ['amount_risk', 'numeric_risk'], 'review_matches': ['1.1%6', '0.290', '20,.000원'], 'review_count': 3, 'penalty_flags': ['brand_distortion', 'odd_spacing', 'odd_word_form'], 'penalty_matches': ['비로키C', '카드시논부가서비스률변경할 수있습니다', '부가서비스률', '계약올', '반영원', '부과하능'], 'penalty_count': 6, 'penalty_score': 30.0}
page methods: Counter({'ocr': 2})
[page 1][ocr]
비로키C 부가서비스 안내 연회비 안내 카드 이용 시 제공되논 모인트 및 할인혜택 등의 부가서비스는 카드 신규 출시(2022년 4월 1일) 이후 3년 이상 축소 폐지없이 유지I니다. 구분 브랜드 총연회비 기본연회비 제유연회비 상기메도불구하고다음과감은사유가발생한경우카드시논부가서비스률변경할 수있습니다 국내전용 BC 5천원 없음 5천원 카드사의 휴업 파산. 경영상의 위기 등에 따른 불가피한 경우 0의2 제류업체의휴업 파산 경영상의위기로인해불가피하

## Retrieval Check
실제 질의를 바꿔가며 검색 품질을 확인합니다.

In [4]:
from chroma.retrieval import CardRetriever

In [5]:
retriever = CardRetriever()
retriever.get_db_info()

INFO:chroma.retrieval:기존 ChromaDB 로드 성공 (데이터 존재)
/home/sms/openclaw_file/project_debugging/PickCardU/src/chroma/retrieval.py:118: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


{'total_chunks': 1564,
 'total_cards': 106,
 'cards': {'BC_Baro_Clear_Plus': 11,
  'BC_Baro_KaPick': 10,
  'BC_BizCorporate': 10,
  'BC_Biz_AirMoney': 8,
  'BC_Business_Sky': 5,
  'BC_Green_v2': 21,
  'BC_KBank_SIMPLE': 9,
  'BC_K_FRIST': 9,
  'BC_ON&OFF': 7,
  'BC_Shopping&': 8,
  'NH_AllWonderful': 14,
  'NH_Namu_NH': 10,
  'NH_Olbareun_Earth': 10,
  'NH_Olbareun_FLEX': 11,
  'NH_Olbareun_OIL&PASS': 10,
  'NH_SKYPASS': 7,
  'NH_SOHO_Dasaroi_OIL': 9,
  'NH_zgm_Self': 9,
  'NH_zgm_living': 13,
  'NH_zgm_shopping': 13,
  'Hana_Everyones_Shinsegae': 2,
  'Hana_JadeClassic': 16,
  'Hana_JadeFirst': 18,
  'Hana_OneMoreNext_Members': 14,
  'Hana_OneStore_1': 3,
  'Hana_One_More_SOHO': 11,
  'Hana_SKYPASS_Amex_Platinum': 2,
  'Hana_Travellog': 9,
  'Hana_Travellog_SKYPASS': 7,
  'Hana_WonderCard2.0': 62,
  'Hyundai_BlueMembers_F_M2_20260323': 25,
  'Hyundai_D_250827': 14,
  'Hyundai_H_250827': 15,
  'Hyundai_M': 19,
  'Hyundai_O_250827': 15,
  'Hyundai_OliveYoung_20260323': 17,
  'Hyundai_S_

In [6]:
test_queries = [
    "편의점 할인 카드",
    "스타벅스 할인 카드",
    "전월실적 30만원 이하 카드",
    "해외 적립 카드",
    "공항라운지 혜택 카드",
]

test_queries

['편의점 할인 카드', '스타벅스 할인 카드', '전월실적 30만원 이하 카드', '해외 적립 카드', '공항라운지 혜택 카드']

In [7]:
search_results = {}

for query in test_queries:
    results = retriever.search_with_score(query, k=3)
    search_results[query] = results
    print("=" * 100)
    print(f"[QUERY] {query}")
    for idx, (doc, score) in enumerate(results, 1):
        print(f"\n[{idx}] score: {score:.4f}")
        print("card_name:", doc.metadata.get("card_name"))
        print("source:", doc.metadata.get("source"))
        print("type:", doc.metadata.get("type"))
        print("quality_status:", doc.metadata.get("quality_status"))
        print("preview:", doc.page_content[:300])
        print("-" * 80)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[QUERY] 편의점 할인 카드

[1] score: 0.7803
card_name: Woori_Classic_TEN
source: Woori_Classic_TEN.pdf
type: clean
quality_status: ready
preview: 예) 1월 31일 이용건 취소 시 2월 2일까지 취소 매출표가 카드사에 접수된 경우에는 1월 이용실적
에서 차감, 이후 접수된 경우는 2월 이용실적에서 차감
• 카드 수령(사용)등록일과 사용개시일 중 빠른 날을 기준으로 다음달 말일까지는 전월실적 조건
미충족 시에도 최저 실적구간 서비스를 제공합니다. (전월실적 조건 충족 시에는 해당 실적구간
서비스를 제공합니다.)
실적 산정 방법 안내
TEN Discount
5대 일상영역 10% 청구할인 음식점 / 주점 및 온라인 간편결제 1% 청구할인
커피 스타벅스, 투썸플레이스, 이디야커피, 
--------------------------------------------------------------------------------

[2] score: 0.7839
card_name: Lotte_LOCA_LIKIT_Eat
source: Lotte_LOCA_LIKIT_Eat.pdf
type: hybrid
quality_status: ready
preview: [page 1][ocr]
UPPER UPPER EAST WEST SIDE SIDE Central Park Times Square LIO CIA LOCA LIKIT Eat ' CHINATOWN WAL STREET 못데카드 ' { ' 움 요

[page 2][native]
LOCA LIKIT Eat
음식점, 배달앱, 커피 60% 할인

[page 3][native]
LOCA LIKIT Eat 혜택 안내
멤버십
쿠팡 로켓와우, 네이버플러스 멤버십
60% 결제일 할인
음식점
ᆞ음식점 업종으로 등록된 가맹점에서 혜택이 제공되며,
주점, 유흥
--------------------------------------------------

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[QUERY] 스타벅스 할인 카드

[1] score: 0.9234
card_name: Samsung_taptap_O
source: Samsung_taptap_O.pdf
type: clean
quality_status: ready
preview: 충전식 선불카드 충전건은 제외됩니다.
* 라이프스타일 할인의 경우 전월 이용금액에서 할인 혜택이 적용된 이용금액
(할인금액이 포함된 총 금액), 국세/지방세/공과금, 아파트 관리비, 대학 등록금,
대중교통, 택시, 상품권 구매, 선불카드 충전건은 제외됩니다.
* 라이프스타일 할인 혜택은 발급월에는 제공되지 않으며, 그 다음 달부터
전월 이용금액 30만원 이상 시 제공됩니다.
라이프스타일 옵션 패키지
옵션(택1) 패키지1 패키지2 패키지3 패키지4 패키지5 패키지6
쇼핑 오픈마켓 7% 할인 1% 적립 1% 적립 7% 할인 1% 적립 
--------------------------------------------------------------------------------

[2] score: 0.9413
card_name: Samsung_5_V4
source: Samsung_5_V4.pdf
type: clean
quality_status: ready
preview: 파스쿠찌, 아티제, 폴 바셋, 블루보틀
제과점 파리바게뜨, 뚜레쥬르, 던킨도너츠
전월 이용금액대별 통합 월 할인한도
50만원 이상 100만원 이상
7,000원 20,000원
* 할인점의 경우, 온라인몰도 포함되며, 기업형 슈퍼마켓(이마트
에브리데이, 롯데슈퍼, 홈플러스 익스프레스 등), 쇼핑 외 결제건
(상품권, 주차장 등), 할인점 내 임대매장은 제외됩니다.
* 해외의 경우, 해외겸용카드에 한해 혜택이 제공됩니다.
* 해외 이용 시 별도의 수수료가 부과됩니다. 자세한 내용은 ‘유의사항’을
확인해 주세요.
* 의료의 경우, 오프라인
--------------------------------------------------------------------

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[QUERY] 해외 적립 카드

[1] score: 0.6904
card_name: BC_Baro_Clear_Plus
source: BC_Baro_Clear_Plus.pdf
type: hybrid
quality_status: review_needed
preview: . 청구됩니다. (단, 해외서비스 수수료는 MasterCard만 적용) 아래의 경우 회원이 이미 남부한 연회비에 반영원 다음의 비용은 반환금액 산정에서 해외 이용 시 청구금액 산출방법은 아래와 같습니다. 제외됩니다: 해외 이용 시 청구금액-(거래미화금액 X 전신환매도물이) + 국제브랜드 수수료2 + -카드의발행 배송등 카드발급신규발급에 소요된 비용 해외서비스 수수료 ( (MasterCard만 적용) -카드이용 시 제공되논 추가적인 혜택 등 부가서비스 제공에 소요된 비용 전신환매도움: 접수일의 비씨카드사 대외결제대행은행의 최초 고시 
--------------------------------------------------------------------------------

[2] score: 0.7153
card_name: IBK_Point3.8(Credit)
source: IBK_Point3.8(Credit).pdf
type: hybrid
quality_status: review_needed
preview: .0%) 저시화다드물 하외이용 수수: : : 기래기화 글o 해외이용 수수: : 울(0.18%) 선신환마노움 비씨카드 대외결제은행은 비씨카드 콤페이지고객센터 해외이용안내)어서 확인 가능합니다. 해외 가망점에시 원화거래(DCC) 시 추가 수수료가 부과되므로 현지통화로 거래하시기 바람니다 해외원화결저서비스(DCC) 사선사단 서비스 이용방법 영업점 콤페이지 및 ONE방크 (카드 카느ON아 해외원화결제에시 신청 가능 해외이용거래는 현지가망점 및 금융기관 사정으로 인하여 접수 /매입이 늦어질 수 있으적, 성구가 지연월 수 있습니다: 해외이용 
--------------------------------

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[QUERY] 공항라운지 혜택 카드

[1] score: 0.8831
card_name: Hyundai_The_Orange_20260330
source: Hyundai_The_Orange_20260330.pdf
type: hybrid
quality_status: ready
preview: [page 4][native]
02
기본 혜택
전월 이용 금액 50만원 이상 시 국내외 가맹점 1% M포인트 적립
· 1회 및 연간 적립 한도 제한 없음
· 신규 발급 시 카드 수령 등록월 다음 달 이용 건까지는 전월 이용 금액 50만원 미만도 혜택 제공
추가 혜택
전월 이용 금액 100만원 이상 시 온라인몰, 다이닝, 웰니스, 테크 영역 10% M포인트 적립
· 대상 영역별 월 1만 M포인트 적립 한도
(본인+가족 카드 합산 기준, 적립 한도 초과 시 기본 혜택 적립률 적용)
· 신규 발급 시 카드 수령 등록월 다음 달 이용 건까
--------------------------------------------------------------------------------

[2] score: 0.8949
card_name: Hyundai_OliveYoung_20260323
source: Hyundai_OliveYoung_20260323.pdf
type: hybrid
quality_status: ready
preview: · 2개 이상 카드사의 카드를 보유할 경우, 보유 카드 정보가 한국신용정보원을
통해 신용카드사간 공유되므로 회원님의 개인신용평점, 이용 한도 등에 영향을
미칠 수 있습니다.
기본·추가 혜택
02
우대 서비스
03
혜택 제공 기준
04
연체 금리
07
카드 이용
유의사항
06
올리브영 현대카드 혜택
--------------------------------------------------------------------------------

[3] score: 0.8979
card_name: Hana_WonderCard2.0
source: Hana_Wond

In [8]:
target_query = "편의점 할인 카드"
search_results.get(target_query, [])

[(Document(metadata={'quality_score': 100.0, 'file_path': '/home/sms/openclaw_file/project_debugging/PickCardU/data/raw/woori/Woori_Classic_TEN.pdf', 'ocr_pages': 0, 'penalty_count': 0, 'penalty_score': 0.0, 'native_quality_score': 100.0, 'source': 'Woori_Classic_TEN.pdf', 'low_quality_pages': 0, 'ocr_quality_score': 0.0, 'card_company': 'woori', 'total_pages': 2, 'type': 'clean', 'review_matches': '[]', 'review_count': 0, 'penalty_flags': '[]', 'card_name': 'Woori_Classic_TEN', 'penalty_matches': '[]', 'review_flags': '[]', 'quality_status': 'ready'}, page_content='예) 1월 31일 이용건 취소 시 2월 2일까지 취소 매출표가 카드사에 접수된 경우에는 1월 이용실적\n에서 차감, 이후 접수된 경우는 2월 이용실적에서 차감\n• 카드 수령(사용)등록일과 사용개시일 중 빠른 날을 기준으로 다음달 말일까지는 전월실적 조건\n미충족 시에도 최저 실적구간 서비스를 제공합니다. (전월실적 조건 충족 시에는 해당 실적구간\n서비스를 제공합니다.)\n실적 산정 방법 안내\nTEN Discount\n5대 일상영역 10% 청구할인 음식점 / 주점 및 온라인 간편결제 1% 청구할인\n커피 스타벅스, 투썸플레이스, 이디야커피, 메 가MGC커피,\n컴포즈커피, 매 머드커피\n※ 스타벅스(사이렌오더) 및 커피브랜드 공식 앱을 통한 결제 건 포함\n※ 커피브랜드의 상품권, 선불카드 구매 / 충전 등의 이용금액은 할인 제외\n※ 백화점/대형할인